### **📚 Librerías**

In [1]:
import requests

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### **🤖 Creación del agente**

In [5]:
# Tool para el clima
@tool("get_weather", description="Get the weather of a city")
def get_weather(city: str) -> str:
    """Get the weather of a city
    
    Args:
        city (str): City name
    
    Returns:
        str: Weather of the city
    """

    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data = response.json()

    latitude = data["results"][0]["latitude"]
    longitude = data["results"][0]["longitude"]
    
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()

    final_response = f"El clima de {city} es {data['current_weather']['temperature']}°C"
    print(final_response)

get_weather.invoke("Bogota")

El clima de Bogota es 12.7°C


In [6]:
# Tool para llamar a la API
@tool("get_products", description="Get the products of the API")
def get_products():
    """Get the products of the API"""
    response = requests.get("https://fakestoreapi.com/products")
    result =  response.json()
    products = " /".join([f"{product['title']}: {product['price']}" for product in result])
    return products

In [7]:
get_products.invoke("")

"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops: 109.95 /Mens Casual Premium Slim Fit T-Shirts : 22.3 /Mens Cotton Jacket: 55.99 /Mens Casual Slim Fit: 15.99 /John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet: 695 /Solid Gold Petite Micropave : 168 /White Gold Plated Princess: 9.99 /Pierced Owl Rose Gold Plated Stainless Steel Double: 10.99 /WD 2TB Elements Portable External Hard Drive - USB 3.0 : 64 /SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s: 109 /Silicon Power 256GB SSD 3D NAND A55 SLC Cache Performance Boost SATA III 2.5: 109 /WD 4TB Gaming Drive Works with Playstation 4 Portable External Hard Drive: 114 /Acer SB220Q bi 21.5 inches Full HD (1920 x 1080) IPS Ultra-Thin: 599 /Samsung 49-Inch CHG90 144Hz Curved Gaming Monitor (LC49HG90DMNXZA) – Super Ultrawide Screen QLED : 999.99 /BIYLACLESEN Women's 3-in-1 Snowboard Jacket Winter Coats: 56.99 /Lock and Love Women's Removable Hooded Faux Leather Moto Biker Jacket: 29.95 /Rain Jacket Women Win

In [8]:
# Instrucciones del sistema
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a 
encontrar los productos que necesitan.

También das información sobre el clima de una ciudad.

Tus herramientas son:
- get_products: Obtiene los productos de la API
- get_weather: Obtiene el clima de una ciudad
"""

message = [
    ("system", system_prompt),
    ("user", "¿Cuál es el clima de Bogota?")
]

In [9]:
# Inicialización del LLM
llm = init_chat_model(
    model = "openai:gpt-4o-mini",
    temperature=0
)

In [10]:
# Agregamos las tools al LLM
llm_with_tools = llm.bind_tools([get_weather, get_products])

In [11]:
# Realizamos el llamado
response = llm_with_tools.invoke(message)

In [12]:
# Vemos las tool utilizadas en la ejecución
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Bogota'},
  'id': 'call_asijCapcGwMnwNaZ1r1PJaHZ',
  'type': 'tool_call'}]